# Bluesky Financial Ticker Co-mention Network — SNA Analysis

**Gruppo:** Buccolini – Fuschi – Ja  
**Dataset:** `bluesky_large_edges_unweighted_10000_1.csv`  
**Graph type:** Simple, undirected, unweighted  

**Sections covered:**
1. Graph loading & basic statistics
2. Connected components analysis
3. Degree distribution analysis (+ power-law fit)
4. Path analysis
5. Clustering coefficient & density
6. Centrality analysis
7. Assortativity
8. Comparison with ER and BA synthetic models

In [ ]:
# ── Dependencies ──────────────────────────────────────────────────────────────
!pip install networkx pandas numpy matplotlib scipy powerlaw -q

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np
import networkx as nx
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
from collections import Counter
import scipy.stats as stats

try:
    import powerlaw
    HAS_POWERLAW = True
except ImportError:
    HAS_POWERLAW = False
    print("powerlaw not available – skipping power-law fit")

%matplotlib inline
plt.rcParams.update({
    'figure.dpi': 110,
    'axes.spines.top': False,
    'axes.spines.right': False,
    'axes.grid': True,
    'grid.alpha': 0.3,
})

SEED = 42
print("All libraries loaded.")

---
## 1. Graph Loading & Basic Statistics

In [ ]:
# ── Load CSV ──────────────────────────────────────────────────────────────────
# Adjust the path if running locally
CSV_PATH = "bluesky_large_edges_unweighted_10000_1.csv"

df_edges = pd.read_csv(CSV_PATH)
print(f"Edge list shape: {df_edges.shape}")
df_edges.head()

In [ ]:
# ── Build graph ───────────────────────────────────────────────────────────────
# Undirected, unweighted (simple graph – multi-edges collapsed)
G = nx.from_pandas_edgelist(df_edges, source='source', target='target')

N = G.number_of_nodes()
E = G.number_of_edges()
avg_deg = 2 * E / N
density = nx.density(G)

print(f"Nodes (N)         : {N:,}")
print(f"Edges (E)         : {E:,}")
print(f"Average degree    : {avg_deg:.4f}")
print(f"Density           : {density:.6f}")
print(f"Self-loops        : {nx.number_of_selfloops(G)}")

---
## 2. Connected Components Analysis

In [ ]:
# ── Component sizes ───────────────────────────────────────────────────────────
components = sorted(nx.connected_components(G), key=len, reverse=True)
num_components = len(components)
gcc_size = len(components[0])
gcc_frac = gcc_size / N

print(f"Number of connected components : {num_components}")
print(f"Giant component (GCC) size     : {gcc_size:,} nodes  ({gcc_frac:.2%} of all nodes)")
print("\nTop-10 component sizes:")
for i, comp in enumerate(components[:10]):
    print(f"  Component {i+1:2d}: {len(comp):,} nodes")

In [ ]:
# ── Component size distribution (excluding GCC) ───────────────────────────────
small_sizes = [len(c) for c in components[1:]]

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

# Bar chart of all component sizes
ax = axes[0]
sizes_all = [len(c) for c in components]
ax.bar(range(len(sizes_all)), sizes_all, color='steelblue', edgecolor='white')
ax.set_xlabel('Component rank')
ax.set_ylabel('Number of nodes')
ax.set_title('Connected component sizes (all)')
ax.set_yscale('log')

# Small components only
ax = axes[1]
if small_sizes:
    size_counts = Counter(small_sizes)
    ax.bar(list(size_counts.keys()), list(size_counts.values()),
           color='coral', edgecolor='white')
    ax.set_xlabel('Component size')
    ax.set_ylabel('Count')
    ax.set_title('Small component sizes (excluding GCC)')

plt.tight_layout()
plt.savefig('components.png', bbox_inches='tight')
plt.show()

In [ ]:
# ── Work on GCC for subsequent analyses ──────────────────────────────────────
GCC = G.subgraph(components[0]).copy()
print(f"GCC — nodes: {GCC.number_of_nodes():,}  edges: {GCC.number_of_edges():,}")

---
## 3. Degree Distribution Analysis

In [ ]:
# ── Degree sequence ───────────────────────────────────────────────────────────
degrees = np.array([d for _, d in G.degree()])

print("Degree statistics (full graph):")
print(f"  min    : {degrees.min()}")
print(f"  max    : {degrees.max()}")
print(f"  mean   : {degrees.mean():.4f}")
print(f"  median : {np.median(degrees):.1f}")
print(f"  std    : {degrees.std():.4f}")
print(f"  skew   : {stats.skew(degrees):.4f}")

In [ ]:
# ── Top-20 highest-degree nodes ───────────────────────────────────────────────
deg_dict = dict(G.degree())
top20 = sorted(deg_dict.items(), key=lambda x: x[1], reverse=True)[:20]
print("Top-20 nodes by degree:")
for ticker, deg in top20:
    print(f"  {ticker:10s}  degree={deg:,}")

In [ ]:
# ── Degree distribution plots ─────────────────────────────────────────────────
deg_count = Counter(degrees)
xs = np.array(sorted(deg_count.keys()))
ys = np.array([deg_count[x] for x in xs])

# CCDF (complementary CDF) — common for heavy tails
sorted_deg = np.sort(degrees)
ccdf = 1 - np.arange(1, len(sorted_deg) + 1) / len(sorted_deg)

fig, axes = plt.subplots(1, 3, figsize=(16, 4))

# Linear histogram
ax = axes[0]
ax.hist(degrees, bins=60, color='steelblue', edgecolor='white', alpha=0.85)
ax.set_xlabel('Degree')
ax.set_ylabel('Count')
ax.set_title('Degree distribution (linear)')
ax.axvline(degrees.mean(), color='red', ls='--', label=f'Mean={degrees.mean():.1f}')
ax.legend()

# Log-log PDF
ax = axes[1]
prob = ys / ys.sum()
ax.loglog(xs, prob, 'o', color='steelblue', alpha=0.7, markersize=4)
ax.set_xlabel('Degree (log)')
ax.set_ylabel('P(k) (log)')
ax.set_title('Degree distribution (log-log PDF)')

# Log-log CCDF
ax = axes[2]
ax.loglog(sorted_deg, ccdf, color='coral', lw=1.5)
ax.set_xlabel('Degree (log)')
ax.set_ylabel('P(K ≥ k) (log)')
ax.set_title('CCDF (log-log)')

plt.tight_layout()
plt.savefig('degree_distribution.png', bbox_inches='tight')
plt.show()

In [ ]:
# ── Power-law fit ─────────────────────────────────────────────────────────────
if HAS_POWERLAW:
    fit = powerlaw.Fit(degrees, discrete=True, verbose=False)
    alpha = fit.power_law.alpha
    xmin  = fit.power_law.xmin
    print(f"Power-law fit:  alpha = {alpha:.4f},  x_min = {xmin}")

    # Compare with log-normal
    R, p = fit.distribution_compare('power_law', 'lognormal')
    print(f"Log-likelihood ratio (power_law vs lognormal): R={R:.4f}, p={p:.4f}")
    winner = 'power_law' if R > 0 else 'lognormal'
    print(f"  → {winner} is the better fit (p={p:.4f})")

    fig, ax = plt.subplots(figsize=(7, 4))
    fit.plot_pdf(color='steelblue', linewidth=2, label='Empirical PDF', ax=ax)
    fit.power_law.plot_pdf(color='red', linestyle='--',
                           label=f'Power-law α={alpha:.2f}', ax=ax)
    fit.lognormal.plot_pdf(color='green', linestyle=':',
                           label='Log-normal', ax=ax)
    ax.set_xlabel('Degree')
    ax.set_ylabel('P(k)')
    ax.set_title('Degree distribution — power-law fit')
    ax.legend()
    plt.tight_layout()
    plt.savefig('powerlaw_fit.png', bbox_inches='tight')
    plt.show()
else:
    print("Skipped: powerlaw library not available")

---
## 4. Path Analysis

In [ ]:
# ── Average shortest path length & diameter (GCC) ────────────────────────────
# For large graphs we sample to estimate average path length
SAMPLE = 500  # number of source nodes for estimation

rng = np.random.default_rng(SEED)
sampled_nodes = rng.choice(list(GCC.nodes()), size=min(SAMPLE, GCC.number_of_nodes()),
                            replace=False)

path_lengths = []
for node in sampled_nodes:
    lengths = nx.single_source_shortest_path_length(GCC, node)
    path_lengths.extend(lengths.values())

path_lengths = np.array(path_lengths)
path_lengths = path_lengths[path_lengths > 0]   # exclude self

avg_path = path_lengths.mean()
print(f"Estimated average shortest path length (GCC, sample={SAMPLE}): {avg_path:.4f}")

# Diameter: exact only feasible on GCC if not too large
# We use a BFS-based approximation (eccentricities from the sample)
max_path = path_lengths.max()
print(f"Observed maximum path length (estimated lower bound): {max_path}")

In [ ]:
# ── Path length distribution ──────────────────────────────────────────────────
pl_counts = Counter(path_lengths)
pl_xs = sorted(pl_counts.keys())
pl_ys = [pl_counts[x] / len(path_lengths) for x in pl_xs]

fig, ax = plt.subplots(figsize=(7, 4))
ax.bar(pl_xs, pl_ys, color='steelblue', edgecolor='white')
ax.axvline(avg_path, color='red', ls='--', label=f'Mean ≈ {avg_path:.2f}')
ax.set_xlabel('Shortest path length')
ax.set_ylabel('Fraction of pairs')
ax.set_title('Shortest path length distribution (GCC, sampled)')
ax.legend()
plt.tight_layout()
plt.savefig('path_distribution.png', bbox_inches='tight')
plt.show()

In [ ]:
# ── Small-world check ─────────────────────────────────────────────────────────
# For a random graph: L_random ≈ ln(N)/ln(avg_k)
N_gcc = GCC.number_of_nodes()
E_gcc = GCC.number_of_edges()
k_gcc  = 2 * E_gcc / N_gcc
L_er  = np.log(N_gcc) / np.log(k_gcc)
print(f"GCC avg degree              : {k_gcc:.2f}")
print(f"Expected L (ER)             : {L_er:.4f}")
print(f"Observed L (sampled)        : {avg_path:.4f}")
print(f"Ratio L_obs / L_ER          : {avg_path / L_er:.4f}  (≈1 → small-world)")

---
## 5. Clustering Coefficient & Density

In [ ]:
# ── Global & average clustering ───────────────────────────────────────────────
avg_clustering = nx.average_clustering(G)
# Global / transitivity
transitivity   = nx.transitivity(G)

print(f"Average clustering coefficient : {avg_clustering:.6f}")
print(f"Global clustering (transitivity): {transitivity:.6f}")
print(f"Graph density                  : {density:.6f}")

# Expected clustering for ER: p = k/(N-1)
p_er = avg_deg / (N - 1)
print(f"\nExpected clustering (ER)       : {p_er:.6f}")
print(f"Ratio C_obs / C_ER             : {avg_clustering / p_er:.2f}")

In [ ]:
# ── Clustering vs degree ───────────────────────────────────────────────────────
clust_dict  = nx.clustering(G)
clust_by_deg = {}
for node in G.nodes():
    d = G.degree(node)
    clust_by_deg.setdefault(d, []).append(clust_dict[node])

deg_vals  = np.array(sorted(clust_by_deg.keys()))
clust_avg = np.array([np.mean(clust_by_deg[d]) for d in deg_vals])

fig, ax = plt.subplots(figsize=(8, 4))
ax.scatter(deg_vals, clust_avg, s=10, alpha=0.6, color='steelblue')
ax.set_xscale('log')
ax.set_xlabel('Degree (log scale)')
ax.set_ylabel('Average clustering')
ax.set_title('Clustering coefficient vs degree')
ax.axhline(avg_clustering, color='red', ls='--', lw=1,
           label=f'Global avg = {avg_clustering:.4f}')
ax.legend()
plt.tight_layout()
plt.savefig('clustering_vs_degree.png', bbox_inches='tight')
plt.show()

In [ ]:
# ── Clustering distribution ───────────────────────────────────────────────────
clust_vals = np.array(list(clust_dict.values()))
print(f"Clustering — min: {clust_vals.min():.4f}  max: {clust_vals.max():.4f}  "
      f"mean: {clust_vals.mean():.4f}  median: {np.median(clust_vals):.4f}")

fig, ax = plt.subplots(figsize=(7, 4))
ax.hist(clust_vals, bins=50, color='coral', edgecolor='white', alpha=0.85)
ax.axvline(avg_clustering, color='red', ls='--', label=f'Mean={avg_clustering:.3f}')
ax.set_xlabel('Clustering coefficient')
ax.set_ylabel('Count')
ax.set_title('Distribution of local clustering coefficients')
ax.legend()
plt.tight_layout()
plt.savefig('clustering_distribution.png', bbox_inches='tight')
plt.show()

---
## 6. Centrality Analysis

In [ ]:
# ── Degree centrality ─────────────────────────────────────────────────────────
deg_centrality = nx.degree_centrality(G)

top10_deg = sorted(deg_centrality.items(), key=lambda x: x[1], reverse=True)[:10]
print("Top-10 by Degree Centrality:")
for node, val in top10_deg:
    print(f"  {node:10s}  {val:.6f}")

In [ ]:
# ── Closeness centrality (GCC) ────────────────────────────────────────────────
closeness = nx.closeness_centrality(GCC)

top10_close = sorted(closeness.items(), key=lambda x: x[1], reverse=True)[:10]
print("Top-10 by Closeness Centrality (GCC):")
for node, val in top10_close:
    print(f"  {node:10s}  {val:.6f}")

In [ ]:
# ── Betweenness centrality (approximate for large graphs) ────────────────────
# k=500 pivot nodes for approximation
betweenness = nx.betweenness_centrality(GCC, k=500, normalized=True, seed=SEED)

top10_between = sorted(betweenness.items(), key=lambda x: x[1], reverse=True)[:10]
print("Top-10 by Betweenness Centrality (GCC, approx):")
for node, val in top10_between:
    print(f"  {node:10s}  {val:.6f}")

In [ ]:
# ── PageRank ──────────────────────────────────────────────────────────────────
pagerank = nx.pagerank(G, alpha=0.85, max_iter=200, tol=1e-6)

top10_pr = sorted(pagerank.items(), key=lambda x: x[1], reverse=True)[:10]
print("Top-10 by PageRank:")
for node, val in top10_pr:
    print(f"  {node:10s}  {val:.8f}")

In [ ]:
# ── Eigenvector centrality (GCC) ──────────────────────────────────────────────
try:
    eigen = nx.eigenvector_centrality(GCC, max_iter=1000, tol=1e-6)
    top10_eig = sorted(eigen.items(), key=lambda x: x[1], reverse=True)[:10]
    print("Top-10 by Eigenvector Centrality (GCC):")
    for node, val in top10_eig:
        print(f"  {node:10s}  {val:.6f}")
except nx.PowerIterationFailedConvergence:
    eigen = None
    print("Eigenvector centrality did not converge")

In [ ]:
# ── Centrality comparison table ───────────────────────────────────────────────
# Collect all centralities for all GCC nodes
records = []
for node in GCC.nodes():
    records.append({
        'ticker':       node,
        'degree':       deg_dict.get(node, 0),
        'degree_c':     deg_centrality.get(node, 0),
        'closeness':    closeness.get(node, 0),
        'betweenness':  betweenness.get(node, 0),
        'pagerank':     pagerank.get(node, 0),
        'eigenvector':  (eigen.get(node, 0) if eigen else 0),
        'clustering':   clust_dict.get(node, 0),
    })

cent_df = pd.DataFrame(records).set_index('ticker')
cent_df.sort_values('degree', ascending=False).head(15)

In [ ]:
# ── Centrality distributions (side by side) ───────────────────────────────────
fig, axes = plt.subplots(2, 2, figsize=(13, 9))

centrality_metrics = [
    ('degree_c',    'Degree Centrality',     'steelblue'),
    ('closeness',   'Closeness Centrality',  'coral'),
    ('betweenness', 'Betweenness Centrality','mediumseagreen'),
    ('pagerank',    'PageRank',              'orchid'),
]

for ax, (col, title, color) in zip(axes.flat, centrality_metrics):
    vals = cent_df[col].values
    ax.hist(vals, bins=60, color=color, edgecolor='white', alpha=0.85)
    ax.set_title(title)
    ax.set_xlabel('Value')
    ax.set_ylabel('Count')
    ax.axvline(vals.mean(), color='black', ls='--', lw=1,
               label=f'Mean={vals.mean():.4f}')
    ax.legend(fontsize=8)

plt.suptitle('Centrality Distributions', fontsize=14, y=1.01)
plt.tight_layout()
plt.savefig('centrality_distributions.png', bbox_inches='tight')
plt.show()

In [ ]:
# ── Centrality bar-chart for top-20 nodes ─────────────────────────────────────
top20_nodes = cent_df.nlargest(20, 'degree')

fig, axes = plt.subplots(2, 2, figsize=(16, 10))

for ax, (col, title, color) in zip(axes.flat, centrality_metrics):
    sub = top20_nodes[col].sort_values(ascending=False)
    ax.barh(sub.index[::-1], sub.values[::-1], color=color, edgecolor='white')
    ax.set_title(f'Top-20 — {title}')
    ax.set_xlabel('Score')

plt.suptitle('Centrality Scores — Top-20 nodes by degree', fontsize=14, y=1.01)
plt.tight_layout()
plt.savefig('centrality_top20.png', bbox_inches='tight')
plt.show()

---
## 7. Assortativity

In [ ]:
# ── Degree assortativity coefficient ─────────────────────────────────────────
r = nx.degree_assortativity_coefficient(G)
print(f"Degree assortativity coefficient r = {r:.6f}")
if r > 0:
    print("  → Assortative: high-degree nodes tend to connect with other high-degree nodes")
elif r < 0:
    print("  → Disassortative: high-degree nodes tend to connect with low-degree nodes")
else:
    print("  → No degree-degree correlation")

In [ ]:
# ── Average neighbor degree vs degree (k_nn plot) ─────────────────────────────
knn = nx.average_neighbor_degree(G)
knn_by_deg = {}
for node, knn_val in knn.items():
    d = G.degree(node)
    knn_by_deg.setdefault(d, []).append(knn_val)

knn_deg = sorted(knn_by_deg.keys())
knn_mean = [np.mean(knn_by_deg[d]) for d in knn_deg]

fig, ax = plt.subplots(figsize=(7, 4))
ax.scatter(knn_deg, knn_mean, s=8, alpha=0.6, color='steelblue')
ax.set_xscale('log')
ax.set_xlabel('Degree k (log)')
ax.set_ylabel('Average neighbor degree k_nn')
ax.set_title(f'k_nn(k) plot  (r = {r:.4f})')
plt.tight_layout()
plt.savefig('assortativity_knn.png', bbox_inches='tight')
plt.show()

---
## 8. Comparison with ER and BA Synthetic Models

In [ ]:
# ── Generate ER and BA graphs matching N and E of the real network ────────────
print(f"Real graph: N={N}, E={E}, <k>={avg_deg:.4f}")

# ER: p = E / (N*(N-1)/2)
p_er_model = E / (N * (N - 1) / 2)
G_ER = nx.erdos_renyi_graph(N, p_er_model, seed=SEED)

# BA: m = round(<k>/2)  → ensures E ≈ E_real
m_ba = max(1, round(avg_deg / 2))
G_BA = nx.barabasi_albert_graph(N, m_ba, seed=SEED)

print(f"ER  graph: N={G_ER.number_of_nodes()}, E={G_ER.number_of_edges()}, "
      f"<k>={2*G_ER.number_of_edges()/G_ER.number_of_nodes():.4f}")
print(f"BA  graph: N={G_BA.number_of_nodes()}, E={G_BA.number_of_edges()}, "
      f"<k>={2*G_BA.number_of_edges()/G_BA.number_of_nodes():.4f}")

In [ ]:
# ── Helper: compute all stats for a graph ─────────────────────────────────────
def compute_stats(Gx, name, sample=300):
    n  = Gx.number_of_nodes()
    e  = Gx.number_of_edges()
    dk = np.array([d for _, d in Gx.degree()])
    avg_k = dk.mean()
    dens  = nx.density(Gx)
    cc_avg = nx.average_clustering(Gx)
    trans  = nx.transitivity(Gx)
    assort = nx.degree_assortativity_coefficient(Gx)
    n_comp = nx.number_connected_components(Gx)
    gcc_n  = len(max(nx.connected_components(Gx), key=len))
    
    # Approximate avg path on GCC
    Gx_gcc = Gx.subgraph(max(nx.connected_components(Gx), key=len)).copy()
    rng2 = np.random.default_rng(SEED)
    src = rng2.choice(list(Gx_gcc.nodes()), size=min(sample, Gx_gcc.number_of_nodes()),
                      replace=False)
    pls = []
    for s in src:
        ls = nx.single_source_shortest_path_length(Gx_gcc, s)
        pls.extend(v for v in ls.values() if v > 0)
    avg_pl = np.mean(pls) if pls else float('nan')
    
    return {
        'Graph': name, 'N': n, 'E': e, '<k>': round(avg_k, 4),
        'Density': round(dens, 6), 'Avg Clustering': round(cc_avg, 6),
        'Transitivity': round(trans, 6), 'Assortativity': round(assort, 6),
        '#Components': n_comp, 'GCC size': gcc_n,
        'Avg path (sample)': round(avg_pl, 4),
    }

stats_real = compute_stats(G,    'Real (Bluesky)')
stats_er   = compute_stats(G_ER, 'ER')
stats_ba   = compute_stats(G_BA, 'BA')

comparison_df = pd.DataFrame([stats_real, stats_er, stats_ba]).set_index('Graph')
comparison_df

In [ ]:
# ── Degree distributions — Real vs ER vs BA ───────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(16, 4))

graph_configs = [
    (G,    'Real (Bluesky)', 'steelblue'),
    (G_ER, 'ER',             'coral'),
    (G_BA, 'BA',             'mediumseagreen'),
]

for ax, (Gx, label, color) in zip(axes, graph_configs):
    dk = np.array([d for _, d in Gx.degree()])
    sorted_dk = np.sort(dk)
    ccdf = 1 - np.arange(1, len(sorted_dk)+1) / len(sorted_dk)
    ax.loglog(sorted_dk, ccdf, color=color, lw=2)
    ax.set_xlabel('Degree (log)')
    ax.set_ylabel('P(K≥k) (log)')
    ax.set_title(f'{label} — CCDF')

plt.suptitle('Degree distribution CCDF: Real vs ER vs BA', y=1.02, fontsize=13)
plt.tight_layout()
plt.savefig('comparison_ccdf.png', bbox_inches='tight')
plt.show()

In [ ]:
# ── Overlay degree distributions ──────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(8, 5))

for Gx, label, color in graph_configs:
    dk = np.array([d for _, d in Gx.degree()])
    sorted_dk = np.sort(dk)
    ccdf = 1 - np.arange(1, len(sorted_dk)+1) / len(sorted_dk)
    ax.loglog(sorted_dk, ccdf, lw=2, label=label, color=color)

ax.set_xlabel('Degree k (log scale)')
ax.set_ylabel('P(K ≥ k) (log scale)')
ax.set_title('Degree CCDF comparison: Real vs ER vs BA')
ax.legend()
plt.tight_layout()
plt.savefig('comparison_ccdf_overlay.png', bbox_inches='tight')
plt.show()

In [ ]:
# ── Clustering vs degree — Real vs ER vs BA ───────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(16, 4))

for ax, (Gx, label, color) in zip(axes, graph_configs):
    clust_x = nx.clustering(Gx)
    cbd = {}
    for node, cv in clust_x.items():
        d = Gx.degree(node)
        cbd.setdefault(d, []).append(cv)
    xs_d = sorted(cbd.keys())
    ys_c = [np.mean(cbd[d]) for d in xs_d]
    ax.scatter(xs_d, ys_c, s=8, alpha=0.5, color=color)
    ax.set_xscale('log')
    ax.set_xlabel('Degree (log)')
    ax.set_ylabel('Avg clustering')
    ax.set_title(f'{label}')

plt.suptitle('Clustering vs Degree: Real vs ER vs BA', y=1.02, fontsize=13)
plt.tight_layout()
plt.savefig('comparison_clustering.png', bbox_inches='tight')
plt.show()

In [ ]:
# ── Bar-chart comparison of key metrics ───────────────────────────────────────
metrics  = ['<k>', 'Density', 'Avg Clustering', 'Transitivity', 'Assortativity']
labels   = ['Real', 'ER', 'BA']
colors   = ['steelblue', 'coral', 'mediumseagreen']
rows     = [stats_real, stats_er, stats_ba]

fig, axes = plt.subplots(1, len(metrics), figsize=(18, 4))

for ax, metric in zip(axes, metrics):
    vals = [row[metric] for row in rows]
    bars = ax.bar(labels, vals, color=colors, edgecolor='white')
    ax.set_title(metric, fontsize=11)
    ax.set_ylabel('Value')
    for bar, v in zip(bars, vals):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() * 1.01,
                f'{v:.4f}', ha='center', va='bottom', fontsize=8)

plt.suptitle('Key metrics comparison: Real vs ER vs BA', fontsize=13, y=1.02)
plt.tight_layout()
plt.savefig('comparison_metrics.png', bbox_inches='tight')
plt.show()

---
## 9. Summary

In [ ]:
print("=" * 65)
print("  BLUESKY CO-MENTION NETWORK — SNA SUMMARY")
print("=" * 65)
print(f"  Nodes                      : {N:,}")
print(f"  Edges                      : {E:,}")
print(f"  Density                    : {density:.6f}")
print(f"  Average degree <k>         : {avg_deg:.4f}")
print(f"  Max degree                 : {degrees.max():,}")
print()
print(f"  Connected components       : {nx.number_connected_components(G)}")
print(f"  GCC fraction               : {gcc_frac:.2%}")
print()
print(f"  Avg clustering coefficient : {avg_clustering:.6f}")
print(f"  Transitivity               : {transitivity:.6f}")
print(f"  Degree assortativity r     : {r:.6f}")
print()
print(f"  Avg shortest path (sample) : {avg_path:.4f}")
print(f"  L_ER expected              : {L_er:.4f}")
print()
print(f"  Top-3 nodes by degree:")
for node, deg in sorted(deg_dict.items(), key=lambda x: x[1], reverse=True)[:3]:
    print(f"    {node}: degree={deg}")
print()
if HAS_POWERLAW:
    print(f"  Power-law exponent α       : {alpha:.4f}")
    print(f"  x_min                      : {xmin}")
print("=" * 65)

print("\nComparison table:")
print(comparison_df.to_string())